In [1]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

print("SIA Wevengers 분석 환경 준비 완료")

SIA Wevengers 분석 환경 준비 완료


In [2]:
# 1. 고위험군 CAMEO 코드 리스트
target_cameo_codes = [
    '150', '151', '152', '153', '154', '155', 
    '190', '191', '192', '193', '194', '195', '196', 
    '200', '201', '202', '203', '204'
]
formatted_codes = ", ".join([f"'{code}'" for code in target_cameo_codes])

query = f"""
SELECT 
    SQLDATE, 
    EventCode,
    GoldsteinScale, 
    NumMentions, 
    AvgTone,
    ActionGeo_Type,
    ActionGeo_Lat, 
    ActionGeo_Long, 
    SOURCEURL
FROM `gdelt-bq.full.events`
WHERE SQLDATE >= 20130401 
  AND (
    (Actor1CountryCode = 'CHN' AND Actor2CountryCode = 'TWN') OR 
    (Actor1CountryCode = 'TWN' AND Actor2CountryCode = 'CHN')
  )
  AND EventCode IN ({formatted_codes})
  AND IsRootEvent = 1                 -- 핵심 사건만 필터링
  AND ActionGeo_Type IN (3, 4, 5)
"""

project_id = "project-fe69a478-943f-4a4a-bc6" # 로그인창 연결됨
df = pandas_gbq.read_gbq(query, project_id=project_id)

# 결과 확인
print(f"2013년부터 2026년 현재까지 총 {len(df)}건의 데이터를 불러왔습니다.(필터링 적용 O)")
display(df.head())

Downloading: 100%|██████████|
2013년부터 2026년 현재까지 총 11900건의 데이터를 불러왔습니다.(필터링 적용 O)


,SQLDATE,EventCode,GoldsteinScale,NumMentions,AvgTone,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,20160910,190,-10.0,3,-4.699739,5,41.0000,123.000,http://www.bangkokpost.com/news/asia/1082960/t...
1,20160910,190,-10.0,4,-4.699739,5,41.0000,123.000,http://www.bangkokpost.com/news/asia/1082960/t...
2,20160702,194,-10.0,10,-5.000000,4,15.0000,115.000,http://www.miragenews.com/man-killed-after-tai...
3,20160702,194,-10.0,30,-3.785108,4,39.9289,116.388,http://www.newsnow.in/news/taiwan-mistakenly-f...
4,20160702,150,-7.2,64,-4.290429,4,39.9289,116.388,http://www.shanghainews.net/index.php/sid/2454...


In [3]:
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'], format='%Y%m%d')

In [4]:
# 1. 전체 날짜 범위 생성
all_dates = pd.date_range(start=df['SQLDATE'].min(), end=df['SQLDATE'].max())

# 2. 사건 개수가 아닌 '기사 언급량(NumMentions)'의 합계로 계산
ts_data = df.groupby('SQLDATE')['NumMentions'].sum().reindex(all_dates, fill_value=0).to_frame(name='event_count')

# 3. 기초 통계량 재계산
avg_events = ts_data['event_count'].mean()
median_events = ts_data['event_count'].median()
std_events = ts_data['event_count'].std()

# 제대로 합산되었는지 검증하기 위한 최대값 확인
max_events = ts_data['event_count'].max()

print(f"최대 일일 기사 언급량: {max_events:.0f} (4595가 나오면 성공!)")
print(f"평균 일일 기사 언급량: {avg_events:.2f}")
print(f"중위수 일일 기사 언급량: {median_events:.2f}")
print(f"표준편차: {std_events:.2f}")

최대 일일 기사 언급량: 4595 (4595가 나오면 성공!)
평균 일일 기사 언급량: 33.05
중위수 일일 기사 언급량: 1.00
표준편차: 158.17


In [5]:
# 결측치 없는 이동평균선 계산
# min_periods=1을 넣어 분석 기간 첫날부터 단 하루도 누락 없이 계산
ts_data['MA7'] = ts_data['event_count'].rolling(window=7, min_periods=1).mean()
ts_data['MA14'] = ts_data['event_count'].rolling(window=14, min_periods=1).mean()
ts_data['MA30'] = ts_data['event_count'].rolling(window=30, min_periods=1).mean()

In [6]:
# 이동평균선 상회(Spike) 일수 계산
# 조건 1: 오늘 기사량이 이동평균선보다 클 것
# 조건 2: 오늘 기사량이 0건은 아닐 것 (첫 번째 코드의 안전장치)
over_ma7 = ts_data[(ts_data['event_count'] > ts_data['MA7']) & (ts_data['event_count'] > 0)]
over_ma14 = ts_data[(ts_data['event_count'] > ts_data['MA14']) & (ts_data['event_count'] > 0)]
over_ma30 = ts_data[(ts_data['event_count'] > ts_data['MA30']) & (ts_data['event_count'] > 0)]
strong_spikes = ts_data[(ts_data['event_count'] > ts_data['MA7']) & (ts_data['event_count'] > ts_data['MA14']) & (ts_data['event_count'] > ts_data['MA30']) & (ts_data['event_count'] > 0)]

count_ma7 = len(over_ma7)
count_ma14 = len(over_ma14)
count_ma30 = len(over_ma30)
count_strong = len(strong_spikes)
total_days = len(ts_data)

In [10]:
# 결과를 DataFrame으로 보기 좋게 정리
summary_df = pd.DataFrame({
    '기준선 (Baseline)': [
        '7일 이동평균 상회 (단기 스파이크)',
        '14일 이동평균 상회 (단기 스파이크)', 
        '30일 이동평균 상회 (장기 스파이크)', 
        '7,14,30일 동시 상회 (강한 스파이크)'
    ],
    '돌파 일수 (Days)': [
        count_ma7,
        count_ma14, 
        count_ma30, 
        count_strong
    ],
    '전체 대비 비율 (%)': [
        round((count_ma7 / total_days) * 100, 2),
        round((count_ma14 / total_days) * 100, 2),
        round((count_ma30 / total_days) * 100, 2),
        round((count_strong / total_days) * 100, 2)
    ]
})

# 인덱스를 1부터 시작하도록 깔끔하게 정리 (선택 사항)
summary_df.index = summary_df.index + 1

# 최종 출력
print(f"--- 이동평균선 상회(Spike) 분석 결과 (총 분석 기간: {total_days}일) ---")
display(summary_df)

--- 이동평균선 상회(Spike) 분석 결과 (총 분석 기간: 4788일) ---


,기준선 (Baseline),돌파 일수 (Days),전체 대비 비율 (%)
1,7일 이동평균 상회 (단기 스파이크),1331,27.80
2,14일 이동평균 상회 (단기 스파이크),1213,25.33
3,30일 이동평균 상회 (장기 스파이크),1082,22.60
4,"7,14,30일 동시 상회 (강한 스파이크)",868,18.13
